# Unmatched Win Matrix Pipeline

Notebook version of the win-matrix merge script, organized into chapters.

## Chapter 1: Imports and Configuration

In [1]:
import pandas as pd
import json
import math
import re
from collections import defaultdict
import os

# In a notebook, getcwd() gets the directory where the notebook is currently running
notebook_dir = os.getcwd()

# Get the parent directory of the notebook's directory
# parent_dir = os.path.dirname(notebook_dir)
parent_dir = r"C:\Users\Moudimash99\Documents\GitHub\unmatched_matcher"
# Build your paths
EXCEL_FILE = os.path.join(parent_dir, "input", "Unmatched_Stats.xlsx")
FIGHTERS_FILE = os.path.join(parent_dir, "input", "fighters.json")

# Optional: Change the working directory if your later code relies on it
os.chdir(parent_dir)

## Chapter 2: Load Fighters and Build Name-to-ID Map

In [2]:
with open(FIGHTERS_FILE, "r", encoding="utf-8") as f:
    fighters_data = json.load(f)

# Map from fighter display name -> id, e.g. "King Arthur" -> "king_arthur"
FIGHTER_NAME_TO_ID = {
    f["name"]: f["id"]
    for f in fighters_data["fighters"]
}

len(FIGHTER_NAME_TO_ID)

60

## Chapter 3: Parsing and Deck Name Normalization Helpers

In [3]:
def normalize_main_col(name):
    """Strip the '\\n(Click to sort Ascending)' junk from column names."""
    if isinstance(name, str):
        return name.split("\n")[0].strip()
    return name


def parse_win_cell(x):
    """
    Parse a Win% cell.
    - Returns float(0-100) if real value.
    - Returns None for empty, '-', or the sentinel -2 (string or numeric).
    """
    try:
        if x is None:
            return None

        if isinstance(x, str):
            s = x.strip()
            if s == "" or s == "-":
                return None
            if s == "-2":
                return None
            s = s.replace("%", "").strip()
            v = float(s)
        else:
            v = float(x)

        if math.isnan(v):
            return None
        if v == -2:
            return None

        return v
    except Exception:
        return None


# Manual aliasing from Excel deck names -> fighter display names
NAME_ALIAS_TO_FIGHTER_NAME = {
    # Buffy variants
    "Buffy (Giles)": "Buffy",
    "Buffy (Xander)": "Buffy",
    "Buffy Giles": "Buffy",
    "Buffy Xander": "Buffy",

    # Cloak & Dagger variants
    "Cloak & Dagger": "Cloak and Dagger",

    # Dr. Sattler variants
    "Dr. Ellie Sattler": "Dr. Sattler",

    # Houdini variants
    "Harry Houdini": "Houdini",

    # Shakespeare variants
    "Shakespeare": "William Shakespeare",

    # Philippa / Golden Bat variants
    "Philippa": "Philippa Eilhart",
    "Golden Bat": "The Golden Bat",
    "InGen" : "Robert Muldoon",

    # Witcher Yennefer/Triss variants
    "Triss (Yennefer)": "Yennefer & Triss",
    "Yennefer (Triss)": "Yennefer & Triss",
    "Triss": "Yennefer & Triss",
    "Yennefer": "Yennefer & Triss",
}


def slugify(name: str) -> str:
    """
    Fallback slug for decks that don't match any fighter.
    Example: 'Blackbeard - Alt 1.0' -> 'blackbeard_alt_1_0'
    """
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def deck_to_fighter_id(raw_name: str) -> str:
    """
    Take a raw Excel deck name and map it to a canonical fighter id.
    Steps:
      1) Strip '\\n(Click to sort Ascending)' and whitespace.
      2) Strip ' - Alt ...' suffix.
      3) Apply NAME_ALIAS_TO_FIGHTER_NAME for known variants.
      4) If final display name matches a fighter, return that id.
      5) Otherwise, return a slug.
    """
    if raw_name is None:
        return None
    name = str(raw_name).strip()
    if name == "":
        return None

    # Remove Excel sort-suffix just in case
    name = normalize_main_col(name)

    # Remove alt suffixes like " - Alt 1.0", " - Alt 2.0", etc.
    name = re.sub(r"\s+-\s*Alt.*$", "", name).strip()

    # Apply manual alias to fighter display name
    fighter_display_name = NAME_ALIAS_TO_FIGHTER_NAME.get(name, name)

    # Try to map to fighter id
    fighter_id = FIGHTER_NAME_TO_ID.get(fighter_display_name)
    if fighter_id is not None:
        return fighter_id

    # Fallback: slug
    return slugify(name)

## Chapter 4: Load the Four Excel Matrices

In [4]:
def load_matrices(excel_file):
    xls = pd.ExcelFile(excel_file)

    # Overall Games Played
    gp_main_raw = pd.read_excel(xls, "UM Cards Games Played", index_col=0)
    gp_main = gp_main_raw[gp_main_raw.index.map(lambda x: isinstance(x, str))].copy()
    gp_main.columns = [normalize_main_col(c) for c in gp_main.columns]

    # Keep only deck-vs-deck columns (drop 'Total' etc.)
    deck_names_main = set(gp_main.index)
    gp_main = gp_main.loc[:, [c for c in gp_main.columns if c in deck_names_main]]

    # Overall Win Percentage
    wp_main_raw = pd.read_excel(xls, "UM Cards Win Percentage", index_col=0)
    wp_main = wp_main_raw[wp_main_raw.index.map(lambda x: isinstance(x, str))].copy()
    wp_main.columns = [normalize_main_col(c) for c in wp_main.columns]
    wp_main = wp_main.loc[wp_main.index.intersection(deck_names_main),
                          [c for c in wp_main.columns if c in deck_names_main]]

    # UMLeague Games Played
    gp_um = pd.read_excel(xls, "UMLeague Games Played", index_col=0)

    # UMLeague Win Percentage
    wp_um = pd.read_excel(xls, "UMLeague Win Percentage", index_col=0)

    return gp_main, wp_main, gp_um, wp_um

In [5]:
gp_main, wp_main, gp_um, wp_um = load_matrices(EXCEL_FILE)



In [6]:
gp_main.head()


,Alice,King Arthur,Medusa,Sinbad,Bigfoot,Robin Hood,InGen,Raptors,Dracula,Invisible Man,...,Titania,William Shakespeare,Ancient Leshen,Ciri,Eredin,Geralt of Rivia,Philippa,Triss (Yennefer),Yennefer (Triss),Bruce Lee
Deck,,,,,,,,,,,,,,,,,,,,,
Alice,0,378,283,353,129,138,36,61,85,69,...,10,6,5,3,1,2,3,0,0,71
King Arthur,378,0,342,251,127,111,18,21,58,72,...,16,2,2,1,0,1,1,0,1,77
Medusa,283,342,1,318,189,192,40,50,104,49,...,6,5,2,6,4,4,4,2,10,86
Sinbad,353,251,318,2,129,115,44,36,97,91,...,3,4,7,5,5,5,1,1,8,81
Bigfoot,129,127,189,129,3,519,53,55,86,56,...,2,4,17,5,11,12,2,2,13,86


## Chapter 5: Merge Sources Into Fighter-ID-Keyed Matrices

In [7]:
def merge_sources_to_fighter_ids(gp_main, wp_main, gp_um, wp_um):
    """
    Aggregate both sources directly into fighter-id keyed matrices.
    UMLeague is treated as loss% and converted to win%.
    """
    merged_games = defaultdict(lambda: defaultdict(int))
    win_num = defaultdict(lambda: defaultdict(float))
    win_den = defaultdict(lambda: defaultdict(int))

    def to_percent(w):
        if w is None:
            return None
        return w * 100.0 if 0.0 <= w <= 1.0 else w

    def accumulate_from_source(gp, wp, invert=False):
        for row_name in gp.index:
            f1 = deck_to_fighter_id(row_name)
            if f1 is None:
                continue

            for col_name in gp.columns:
                f2 = deck_to_fighter_id(col_name)
                if f2 is None:
                    continue

                g = gp.at[row_name, col_name]
                g = 0 if pd.isna(g) else int(g)

                merged_games[f1][f2] += g

                w = None
                if (row_name in wp.index) and (col_name in wp.columns):
                    w = parse_win_cell(wp.at[row_name, col_name])

                if w is not None and g > 0:
                    pct = to_percent(w)
                    actual_win_pct = 100.0 - pct if invert else pct
                    win_num[f1][f2] += actual_win_pct * g
                    win_den[f1][f2] += g

    accumulate_from_source(gp_main, wp_main, invert=False)
    accumulate_from_source(gp_um, wp_um, invert=True)

    merged_win_pct = defaultdict(dict)
    all_fighter_ids = set(FIGHTER_NAME_TO_ID.values()) | set(merged_games.keys())

    for f1 in all_fighter_ids:
        if f1 not in merged_games:
            merged_games[f1] = {}
        if f1 not in merged_win_pct:
            merged_win_pct[f1] = {}

        for f2 in all_fighter_ids:
            merged_games[f1][f2] = merged_games[f1].get(f2, 0)

            if win_den[f1][f2] == 0:
                merged_win_pct[f1][f2] = -2
            else:
                merged_win_pct[f1][f2] = win_num[f1][f2] / win_den[f1][f2]

    return merged_games, merged_win_pct


merged_games, merged_win_pct = merge_sources_to_fighter_ids(
    gp_main, wp_main, gp_um, wp_um
)


## Chapter 6: Execute Pipeline and Export JSON

In [8]:
gp_main, wp_main, gp_um, wp_um = load_matrices(EXCEL_FILE)
merged_games, merged_win_pct = merge_sources_to_fighter_ids(
    gp_main, wp_main, gp_um, wp_um
)

with open("input/merged_games.json", "w", encoding="utf-8") as f:
    json.dump(merged_games, f, indent=2, ensure_ascii=False)

with open("input/merged_win_pct.json", "w", encoding="utf-8") as f:
    json.dump(merged_win_pct, f, indent=2, ensure_ascii=False)

print("Wrote input/merged_games.json and input/merged_win_pct.json")

Wrote input/merged_games.json and input/merged_win_pct.json


In [9]:
# get me win% of elektra vs winter soldier
f1 = "elektra"
f2 = "winter_soldier"

print(f"Win% of {f1} vs {f2}: {merged_win_pct[f1][f2]:.2f}% over {merged_games[f1][f2]} games")

def collect_source_cells(gp, wp, source_name, target_row_id, target_col_id):
    rows = []
    for r in gp.index:
        if deck_to_fighter_id(r) != target_row_id:
            continue
        for c in gp.columns:
            if deck_to_fighter_id(c) != target_col_id:
                continue

            g = gp.at[r, c]
            g = 0 if pd.isna(g) else int(g)

            w_raw = wp.at[r, c] if (r in wp.index and c in wp.columns) else None
            w = parse_win_cell(w_raw)

            rows.append({
                "source": source_name,
                "row_name": r,
                "col_name": c,
                "games": g,
                "win_raw": w_raw,
                "win_parsed": w,
            })
    return rows


def to_percent(w):
    if w is None:
        return None
    return w * 100.0 if 0.0 <= w <= 1.0 else w


rows = []
rows += collect_source_cells(gp_main, wp_main, "main", f1, f2)
rows += collect_source_cells(gp_um, wp_um, "umleague", f1, f2)

verify_df = pd.DataFrame(rows)
display(verify_df)

total_games_original = int(verify_df["games"].sum()) if not verify_df.empty else 0
usable = verify_df[(verify_df["win_parsed"].notna()) & (verify_df["games"] > 0)]

if usable.empty:
    original_weighted_win_pct = -2
else:
    weighted = usable["win_parsed"].map(to_percent)
    source_adjusted = [100.0 - v if row["source"] == "umleague" else v for v, (_, row) in zip(weighted, usable.iterrows())]
    original_weighted_win_pct = sum(v * g for v, g in zip(source_adjusted, usable["games"])) / sum(usable["games"])

print(f"Original weighted win% ({f1} vs {f2}): {original_weighted_win_pct:.2f}% over {total_games_original} games")
print(f"Merged   weighted win% ({f1} vs {f2}): {merged_win_pct[f1][f2]:.2f}% over {merged_games[f1][f2]} games")

is_match = (
    total_games_original == merged_games[f1][f2]
    and abs(original_weighted_win_pct - merged_win_pct[f1][f2]) < 1e-9
)
print("Match:", is_match)


Win% of elektra vs winter_soldier: 65.43% over 23 games


,source,row_name,col_name,games,win_raw,win_parsed
0,main,Elektra,Winter Soldier,12,0.75,0.75
1,umleague,Elektra,Winter Soldier,11,0.45,0.45


Original weighted win% (elektra vs winter_soldier): 65.43% over 23 games
Merged   weighted win% (elektra vs winter_soldier): 65.43% over 23 games
Match: True
